# Trial Analysis: Phase 1 - Event Timeline Generation

This notebook parses the raw event CSV generated by `bag_to_event_csv.py` to identify the precise start, end, and intermediate marked events for each trial. The output is a structured YAML file containing this timeline, which serves as the input for subsequent analysis phases.

## 1. Imports, Configuration, and Helper Functions

In [9]:
import pandas as pd
import numpy as np
import yaml
from pathlib import Path

# --- Variables to Configure ---
cohort_number = 4
input_csv_file = Path(f'../../data/cohort-{cohort_number}-raw.csv')
output_event_timeline_file = Path(f'../../data/cohort-{cohort_number}-event-timeline.yaml')

# --- Constants ---
# Time threshold to group button press events. If the time between two 'True' 
# events is greater than this, they are considered separate logical events.
EVENT_GROUPING_THRESHOLD_NS = 75_000_000  # 75 milliseconds

# --- Helper Functions ---
def robust_csv_value_to_bool(value) -> bool:
    """Converts a value read from a CSV cell to a boolean robustly."""
    if pd.isna(value):
        return False
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return bool(value)
    return str(value).strip().lower() == 'true'

def process_and_finalize_event_group(group_ts_list: list) -> int:
    """Calculates the average timestamp of an event group."""
    if not group_ts_list:
        return 0
    # Calculate the average time of the event group from its first and last timestamp
    return (group_ts_list[0] + group_ts_list[-1]) // 2


## 2. First Pass: Find Trial Boundaries and Marked Events

In [10]:
# Load and prepare the DataFrame
try:
    df = pd.read_csv(input_csv_file, keep_default_na=True, na_values=[''])
    print(f"Successfully loaded {input_csv_file}")
except FileNotFoundError:
    print(f"Error: Input CSV file not found at '{input_csv_file}'")
    df = None

if df is not None:
    # Ensure required columns are present and converted to boolean
    required_events = ['researcher-trial-start', 'researcher-trial-end', 'researcher-mark-event']
    for col in required_events:
        if col not in df.columns:
            raise ValueError(f"Error: Required column '{col}' not found in the input CSV.")
        df[col] = df[col].apply(robust_csv_value_to_bool)
        
    # --- State Machine Initialization ---
    event_timeline = {}
    current_trial_number = 1
    
    # State variables for trial start/end
    active_trial_start_ts = None
    last_start_signal_ts = None
    
    # State variables for marked events within a trial
    active_trial_marked_events = []
    current_marked_event_group_ts = []

    print("Parsing CSV to build event timeline...")
    # --- Main Loop ---
    for index, row in df.iterrows():
        timestamp_ns = row['timestamp']
        
        # --- Trial Start Logic (on button release) ---
        if row['researcher-trial-start']:
            if active_trial_start_ts is not None:
                print(f"Warning (row {index + 2}): New 'start' signal while trial {current_trial_number} was active. Aborting previous trial.")
            last_start_signal_ts = timestamp_ns
            active_trial_start_ts = None # Reset active trial until release

        elif last_start_signal_ts is not None and active_trial_start_ts is None:
            # Start button was just released, a trial is now officially active
            active_trial_start_ts = last_start_signal_ts
            print(f"TRIAL {current_trial_number} START detected at timestamp: {active_trial_start_ts}")
            # Reset accumulators for the new trial
            active_trial_marked_events = []
            current_marked_event_group_ts = []
        
        # --- Logic that runs only during an active trial ---
        if active_trial_start_ts is not None:
            # --- Marked Event Logic (time-based grouping) ---
            if row['researcher-mark-event']:
                if current_marked_event_group_ts and (timestamp_ns - current_marked_event_group_ts[-1]) > EVENT_GROUPING_THRESHOLD_NS:
                    # Gap is too large, finalize old group and start a new one
                    final_event_ts = process_and_finalize_event_group(current_marked_event_group_ts)
                    active_trial_marked_events.append(final_event_ts)
                    print(f"  - Marked event for trial {current_trial_number} recorded at avg time {final_event_ts}")
                    current_marked_event_group_ts = [timestamp_ns]
                else:
                    # Continue the current group
                    current_marked_event_group_ts.append(timestamp_ns)
            
            # --- Trial End Logic (on first button press) ---
            if row['researcher-trial-end']:
                trial_end_ts = timestamp_ns
                print(f"TRIAL {current_trial_number} END detected at timestamp: {trial_end_ts}")

                # Finalize any pending marked event group
                if current_marked_event_group_ts:
                    final_event_ts = process_and_finalize_event_group(current_marked_event_group_ts)
                    active_trial_marked_events.append(final_event_ts)
                    print(f"  - Marked event (held at trial end) for trial {current_trial_number} recorded at avg time {final_event_ts}")

                # --- Assemble the trial data dictionary ---
                trial_data = {
                    'trial-start': {
                        'timestamp': active_trial_start_ts,
                        'trial-rel-time': 0.0
                    }
                }
                
                # Add marked events in the order they appeared
                for i, event_ts in enumerate(sorted(active_trial_marked_events)):
                    rel_time = (event_ts - active_trial_start_ts) / 1e9
                    trial_data[f'event-{i+1}'] = {
                        'timestamp': event_ts,
                        'trial-rel-time': round(rel_time, 4)
                    }
                
                # Add trial end
                rel_time = (trial_end_ts - active_trial_start_ts) / 1e9
                trial_data['trial-end'] = {
                    'timestamp': trial_end_ts,
                    'trial-rel-time': round(rel_time, 4)
                }
                
                event_timeline[str(current_trial_number)] = trial_data
                
                # Reset state for the next trial
                current_trial_number += 1
                active_trial_start_ts = None
                last_start_signal_ts = None
                active_trial_marked_events = []
                current_marked_event_group_ts = []
    
    print("\nParsing complete.")

Successfully loaded ../../data/cohort-4-raw.csv
Parsing CSV to build event timeline...
TRIAL 1 START detected at timestamp: 1748639924929081744
Warning (row 68190): New 'start' signal while trial 1 was active. Aborting previous trial.
TRIAL 1 START detected at timestamp: 1748639924979120999
Warning (row 68196): New 'start' signal while trial 1 was active. Aborting previous trial.
TRIAL 1 START detected at timestamp: 1748639925029141364
Warning (row 68202): New 'start' signal while trial 1 was active. Aborting previous trial.
TRIAL 1 START detected at timestamp: 1748639925079515317
Warning (row 68208): New 'start' signal while trial 1 was active. Aborting previous trial.
TRIAL 1 START detected at timestamp: 1748639925129687240
Warning (row 68214): New 'start' signal while trial 1 was active. Aborting previous trial.
TRIAL 1 START detected at timestamp: 1748639925180100672
Warning (row 68220): New 'start' signal while trial 1 was active. Aborting previous trial.
TRIAL 1 START detected at

## 3. Save and Inspect Event Timeline

In [11]:
if event_timeline:
    # Save the timeline to a YAML file for the next phase of analysis
    try:
        with open(output_event_timeline_file, 'w') as f:
            yaml.dump(event_timeline, f, indent=2, default_flow_style=False, sort_keys=False)
        print(f"Event timeline successfully saved to: {output_event_timeline_file}")
    except Exception as e:
        print(f"Error saving YAML file: {e}")

    # Display the final dictionary for inspection
    # For a cleaner look, you can also use `import json; print(json.dumps(event_timeline, indent=2))`
    print("\n--- Generated Event Timeline ---")
    # event_timeline
else:
    print("No complete trials were found. No timeline generated.")

Event timeline successfully saved to: ../../data/cohort-4-event-timeline.yaml

--- Generated Event Timeline ---


In [12]:
# Generate mode timeline